# Module 5 — Learning Activities 1 & 2
## Data Exploration & Data Cleaning with PySpark — Bank Marketing dataset

This notebook covers **both** Module 5 activities on the UCI **Bank Marketing** dataset (`bank.csv`):

- **Activity 1 — Data Exploration:** numeric statistics (mean/median/std), bar graphs of `previous` (raw + normalized), histograms of `age` (raw + normalized), binned `age` bar chart, and an `age` vs `balance` scatter plot.
- **Activity 2 — Data Cleaning:** remove every row whose `job` is `unknown` or `unemployed`, save the cleaned data, **re-run all of Activity 1** on it, and report the differences.

**How to run:** built for **Google Colab** (Dr. Chen's recommended setup). Run the cells top to bottom. The Activity-1 tasks are written as small reusable functions so Activity 2 can replay them on the cleaned data with one call each.

> Dataset columns: `age, job, marital, education, default, balance, housing, loan, contact, day, month, duration, campaign, pdays, previous, poutcome, y`
> **Numeric attributes:** `age, balance, day, duration, campaign, pdays, previous`.

---
## Part 0 — Setup

### 0.1 Install PySpark
On Colab, PySpark is not pre-installed. `-q` keeps the output quiet.

In [ ]:
!pip install -q pyspark

### 0.2 Locate `bank.csv`
The file lives in the repo at `.../module-05-data-exploration-and-cleaning/activities-dataset-bank-and-marketing/bank/bank.csv`.
This cell looks for it locally; if it is not found (typical on a fresh Colab runtime) it opens an **upload** dialog — pick `bank.csv` from your machine.

In [ ]:
import os

CANDIDATES = [
    "bank.csv",
    "bank/bank.csv",
    "activities-dataset-bank-and-marketing/bank/bank.csv",
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files
        print("bank.csv not found. Upload it now (repo: .../activities-dataset-bank-and-marketing/bank/bank.csv)")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError("Place bank.csv next to this notebook and re-run.")

print("Using:", DATA_PATH)

### 0.3 Create a `SparkSession`
The `SparkSession` is the entry point to Spark DataFrames and Spark SQL (`getOrCreate` reuses one if it already exists).

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("BDA601_Module5_BankMarketing")
    .getOrCreate()
)
spark

### 0.4 Load the CSV into a DataFrame
This dataset is **semicolon-separated** (`;`) with **quoted** string fields, so we set `sep=';'`, `header=True`, and `inferSchema=True` so Spark detects the numeric columns automatically.

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("quote", '"')
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

print("Rows:", df.count(), "| Columns:", len(df.columns))
df.printSchema()
df.show(5)

### 0.5 Define the numeric attributes and the plotting library
We list the numeric columns once and reuse the list everywhere. `matplotlib` runs **on the driver**, so every plot below first reduces the data inside Spark and only pulls a small summary back (this is the Module 4 lesson: never `.collect()` a big DataFrame to the driver).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

NUMERIC = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

---
# Activity 1 — Data Exploration

Each task is wrapped in a small function so we can replay the **whole** activity on the cleaned data in Activity 2.

### 1.1 Mean, median and standard deviation of every numeric attribute
- **mean** and **std** come from a single `.agg(...)` pass.
- **median** is the 50th percentile via `approxQuantile` (exact medians are expensive on big data; `relativeError=0.001` is very accurate and cheap).

In [ ]:
def numeric_summary(frame, label=""):
    aggs = []
    for c in NUMERIC:
        aggs += [F.mean(c).alias(c + "_mean"), F.stddev(c).alias(c + "_std")]
    row = frame.agg(*aggs).collect()[0].asDict()
    medians = {c: frame.approxQuantile(c, [0.5], 0.001)[0] for c in NUMERIC}

    out = pd.DataFrame(
        {
            "mean":   [row[c + "_mean"] for c in NUMERIC],
            "median": [medians[c] for c in NUMERIC],
            "std":    [row[c + "_std"] for c in NUMERIC],
        },
        index=NUMERIC,
    ).round(3)
    print(f"Numeric summary {label}")
    display(out)
    return out

summary_raw = numeric_summary(df, "(raw data)")

### 1.2 Bar graph of `previous`
`previous` = number of contacts before this campaign. We count occurrences of each value in Spark, then draw the bars.

In [ ]:
def plot_previous(frame, label="", normalized=False):
    counts = frame.groupBy("previous").count().orderBy("previous").toPandas()
    y = counts["count"].astype(float)
    if normalized:
        y = y / y.sum()
    plt.figure(figsize=(11, 4))
    plt.bar(counts["previous"], y)
    plt.xlabel("previous (number of prior contacts)")
    plt.ylabel("proportion" if normalized else "count")
    kind = "Normalized bar graph" if normalized else "Bar graph"
    plt.title(f"{kind} of 'previous' {label}")
    plt.tight_layout()
    plt.show()

plot_previous(df, "(raw data)")

### 1.3 Normalized bar graph of `previous`
"Normalized" here = **relative frequency** (each bar divided by the total count, so the bars sum to 1). Same shape as above, but the y-axis is now a proportion — useful for comparing two datasets of different sizes (which is exactly what Activity 2 needs).

In [ ]:
plot_previous(df, "(raw data)", normalized=True)

### 1.4 Histogram of `age`
Spark DataFrames have no native histogram, so we drop to the underlying **RDD** and use `.histogram(n_bins)`. It returns `(bin_edges, counts)` — a tiny summary we plot on the driver.

In [ ]:
def plot_age_hist(frame, label="", normalized=False, bins=20):
    if normalized:
        mn, mx = frame.agg(F.min("age"), F.max("age")).collect()[0]
        series = frame.select(((F.col("age") - mn) / (mx - mn)).alias("v"))
        xlab = "age (min-max normalized to [0, 1])"
    else:
        series = frame.select(F.col("age").alias("v"))
        xlab = "age (years)"

    edges, counts = series.rdd.flatMap(lambda r: r).histogram(bins)
    centers = [(edges[i] + edges[i + 1]) / 2 for i in range(len(counts))]
    width = (edges[1] - edges[0]) * 0.9

    plt.figure(figsize=(11, 4))
    plt.bar(centers, counts, width=width)
    plt.xlabel(xlab)
    plt.ylabel("count")
    kind = "Histogram of normalized 'age'" if normalized else "Histogram of 'age'"
    plt.title(f"{kind} {label}")
    plt.tight_layout()
    plt.show()

plot_age_hist(df, "(raw data)")

### 1.5 Histogram of the normalized `age`
We **min-max normalize** `age` to the range `[0, 1]` (`(age - min) / (max - min)`) and histogram that. The distribution shape is identical to 1.4 — only the x-axis scale changes — which is the point of normalization: comparability across attributes/datasets without changing the underlying pattern.

In [ ]:
plot_age_hist(df, "(raw data)", normalized=True)

### 1.6 Bin `age` and draw a bar chart
We bucket `age` into 10-year bands (`floor(age/10)*10` → `20-29`, `30-39`, ...), count each band in Spark, and plot. Binning (a.k.a. discretization) is also a smoothing / data-reduction technique from the Han reading.

In [ ]:
def plot_age_bins(frame, label="", width=10):
    binned = (
        frame.withColumn("age_bin", (F.floor(F.col("age") / width) * width).cast("int"))
        .groupBy("age_bin").count().orderBy("age_bin")
        .toPandas()
    )
    labels = [f"{b}-{b + width - 1}" for b in binned["age_bin"]]
    plt.figure(figsize=(11, 4))
    plt.bar(labels, binned["count"])
    plt.xlabel(f"age band ({width}-year bins)")
    plt.ylabel("count")
    plt.title(f"Binned 'age' bar chart {label}")
    plt.tight_layout()
    plt.show()
    return binned

_ = plot_age_bins(df, "(raw data)")

### 1.7 Scatter plot — `age` vs `balance`
This dataset is small (~4.5k rows) so we can plot every point. On a big dataset you would `.sample(...)` first (the `sample` argument below does this).

In [ ]:
def plot_age_balance(frame, label="", sample=1.0, seed=42):
    pdf = frame.select("age", "balance")
    if sample < 1.0:
        pdf = pdf.sample(False, sample, seed=seed)
    pdf = pdf.toPandas()
    plt.figure(figsize=(9, 6))
    plt.scatter(pdf["age"], pdf["balance"], alpha=0.3, s=18)
    plt.xlabel("age (years)")
    plt.ylabel("balance (euros)")
    plt.title(f"age vs balance {label}")
    plt.tight_layout()
    plt.show()

plot_age_balance(df, "(raw data)")

### 1.8 Activity 1 — discussion prompt
Write your forum post here. Things worth commenting on (anchor to the numbers above):
- **`age`** is roughly symmetric (mean ≈ median ≈ 41); most clients are 30–49.
- **`balance`** is heavily **right-skewed** (mean ≈ 1423 ≫ median 444, huge std ≈ 3010) — a few rich clients pull the mean up; the **median is the honest "typical" value**.
- **`previous`** is ~0 for the vast majority (median 0): most clients were **not** contacted before this campaign.
- **`pdays` = -1** as a median is a **sentinel**, not a real duration ("never previously contacted") — a data-quality trap to flag, not average blindly.

---
# Activity 2 — Data Cleaning

The data owner decides any row whose **`job`** is `unknown` or `unemployed` is **inaccurate** and must be removed *before* exploration. We then re-run every Activity 1 task on the cleaned data and compare.

### 2.1 Inspect the `job` column first
Always look before you delete. How many rows will each value remove?

In [ ]:
df.groupBy("job").count().orderBy(F.desc("count")).show(20, truncate=False)

to_remove = df.filter(F.col("job").isin("unknown", "unemployed")).count()
print(f"Rows to remove (job in unknown/unemployed): {to_remove}")

### 2.2 Remove the inaccurate rows
`~ ... isin(...)` keeps every row whose `job` is **not** in the blocklist.

In [ ]:
clean_df = df.filter(~F.col("job").isin("unknown", "unemployed"))

print(f"Before: {df.count()} rows")
print(f"After:  {clean_df.count()} rows")
print(f"Removed: {df.count() - clean_df.count()} rows")

### 2.3 Save the cleaned data
Spark writes a **folder** of part-files (it is distributed), so `coalesce(1)` forces a single CSV part. The activity asks for a file named `bank.csv`; here we write `bank_cleaned/` and keep the same `;` delimiter and header. On Colab you can then download the single `part-*.csv` from the file browser (or `rename` it to `bank.csv`).

In [ ]:
(
    clean_df
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .option("sep", ";")
    .csv("bank_cleaned")
)
print("Saved to ./bank_cleaned/  (one part-*.csv inside)")
import os
print([f for f in os.listdir("bank_cleaned") if f.endswith(".csv")])

### 2.4 Re-run **all** of Activity 1 on the cleaned data
Because every Activity 1 task is a function, replaying the whole activity is one call each.

In [ ]:
summary_clean = numeric_summary(clean_df, "(cleaned data)")

In [ ]:
plot_previous(clean_df, "(cleaned data)")
plot_previous(clean_df, "(cleaned data)", normalized=True)

In [ ]:
plot_age_hist(clean_df, "(cleaned data)")
plot_age_hist(clean_df, "(cleaned data)", normalized=True)

In [ ]:
_ = plot_age_bins(clean_df, "(cleaned data)")

In [ ]:
plot_age_balance(clean_df, "(cleaned data)")

### 2.5 Compare raw vs cleaned (numeric summary side by side)

In [ ]:
compare = summary_raw.add_suffix("_raw").join(summary_clean.add_suffix("_clean"))
compare = compare[["mean_raw", "mean_clean", "median_raw", "median_clean", "std_raw", "std_clean"]]
compare["mean_delta"] = (compare["mean_clean"] - compare["mean_raw"]).round(3)
display(compare)

### 2.6 Activity 2 — discussion prompt (what changed and why)
We removed **166 rows (~3.7%)**: 128 `unemployed` + 38 `unknown`. Expected observations:

- **Aggregate statistics barely move** — removing 3.7% of rows leaves mean `age`, `previous`, `campaign`, etc. almost identical. The dataset is large enough that a small, non-extreme slice does not shift the centre.
- **Mean `balance` actually *rises* slightly** (≈ 1423 → 1432) even though we removed clients. That is the interesting bit: **`unemployed` clients skew toward lower balances**, so dropping them nudges the average *up*. A good example of how a cleaning rule can subtly bias a distribution.
- **Medians are essentially unchanged** (`age` 39, `previous` 0, `pdays` -1) — medians are robust to dropping a small, non-outlier group, which is exactly why they are the safer "typical value".
- **Bar/histogram shapes are visually the same** — the cleaning targeted one categorical column (`job`), not the numeric distributions, so the *shape* of `age`/`previous` is preserved; only the counts shrink.

**So what:** the owner's rule is defensible (those rows were declared inaccurate), and crucially it did **not** distort the analysis — but you should *report* the small upward shift in mean balance so a downstream model owner knows the cleaning introduced a mild selection effect.

---
## Appendix — Expected reference values (computed offline)
Use these to check your Spark output. Computed independently with **pandas** on the same `bank.csv` (rounded to 3 dp).

**Activity 1 — raw data (4521 rows):**

| attribute | mean | median | std |
|---|---|---|---|
| age | 41.170 | 39.0 | 10.576 |
| balance | 1422.658 | 444.0 | 3009.638 |
| day | 15.915 | 16.0 | 8.248 |
| duration | 263.961 | 185.0 | 259.857 |
| campaign | 2.794 | 2.0 | 3.110 |
| pdays | 39.767 | -1.0 | 100.121 |
| previous | 0.543 | 0.0 | 1.694 |

**Activity 2 — cleaned data (4355 rows; removed 128 `unemployed` + 38 `unknown` = 166):**

| attribute | mean | median | std |
|---|---|---|---|
| age | 41.117 | 39.0 | 10.583 |
| balance | 1431.762 | 442.0 | 3046.929 |
| day | 15.911 | 16.0 | 8.197 |
| duration | 263.275 | 185.0 | 257.090 |
| campaign | 2.799 | 2.0 | 3.088 |
| pdays | 39.890 | -1.0 | 99.594 |
| previous | 0.545 | 0.0 | 1.698 |

`job` counts (raw): management 969 · blue-collar 946 · technician 768 · admin. 478 · services 417 · retired 230 · self-employed 183 · entrepreneur 168 · **unemployed 128** · housemaid 112 · student 84 · **unknown 38**.